In [0]:
# Load Delta table
df = spark.table("telco_customer_churn")

# Verify it loaded
print(f"Rows: {df.count()}")
df.printSchema()

Rows: 7043
root
 |-- CustomerID: string (nullable = true)
 |-- Count: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zip Code: long (nullable = true)
 |-- Lat Long: string (nullable = true)
 |-- Latitude: decimal(17,15) (nullable = true)
 |-- Longitude: decimal(17,14) (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Senior Citizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Tenure Months: long (nullable = true)
 |-- Phone Service: string (nullable = true)
 |-- Multiple Lines: string (nullable = true)
 |-- Internet Service: string (nullable = true)
 |-- Online Security: string (nullable = true)
 |-- Online Backup: string (nullable = true)
 |-- Device Protection: string (nullable = true)
 |-- Tech Support: string (nullable = true)
 |-- Streaming TV: string (nullable = true)
 |-- Streaming Movies: string (nullable = tr

In [0]:
import pandas as pd
import numpy as np

# Load Delta table as pandas 
df = spark.table("telco_customer_churn").toPandas()
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded: 7043 rows, 33 columns


In [0]:
# Mirrors your schema.py DROP_COLUMNS exactly
DROP_COLUMNS = [
    "CustomerID", "Count", "Country", "State", "City",
    "Zip Code", "Lat Long", "Latitude", "Longitude",
    "Churn Score", "Churn Reason", "CLTV", "Churn Label"
]

# Keep CustomerID separately for predict.py logic later
customer_ids = df["CustomerID"].copy()

# Apply drops
df = df.drop(columns=[col for col in DROP_COLUMNS if col in df.columns])

TARGET_COLUMN = "Churn Value"
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"{TARGET_COLUMN} not found")

print(f"After schema drop: {df.shape[1]} columns")
print(df.columns.tolist())

After schema drop: 20 columns
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']


In [0]:
# Explicitly cast known numeric columns by name — don't rely on dtype detection
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Monthly Charges"] = pd.to_numeric(df["Monthly Charges"], errors="coerce")
df["Tenure Months"] = pd.to_numeric(df["Tenure Months"], errors="coerce")

# Fill numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Fill categorical nulls with mode
for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Binary encode ONLY pure Yes/No columns
for col in df.select_dtypes(include=["object"]).columns.tolist():
    unique_vals = set(df[col].dropna().unique())
    if unique_vals.issubset({"Yes", "No"}):
        df[col] = (df[col] == "Yes").astype(int)

# Encode Gender explicitly (Male=1, Female=0)
df["Gender"] = (df["Gender"] == "Male").astype(int)

# Ensure Churn Value is int
if df["Churn Value"].dtype == "object":
    df["Churn Value"] = (df["Churn Value"] == "Yes").astype(int)

# Confirm what's left as object before get_dummies
remaining_cats = df.select_dtypes(include=["object"]).columns.tolist()
print(f"Columns being one-hot encoded: {remaining_cats}")
# Expected: ['Multiple Lines', 'Internet Service', 'Contract', 'Payment Method']

df = pd.get_dummies(df, columns=remaining_cats, drop_first=True)

# Clean column names for Delta
df.columns = [
    col.replace(" ", "_")
       .replace("(", "")
       .replace(")", "")
       .replace("-", "_")
    for col in df.columns
]

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(df.dtypes)
# Convert bool columns to int for Delta compatibility
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

print("Bool columns converted to int")
print(df.dtypes)

Columns being one-hot encoded: ['Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Payment Method']
Shape: 7043 rows, 31 columns
Missing values: 0
Gender                                     int64
Senior_Citizen                             int64
Partner                                    int64
Dependents                                 int64
Tenure_Months                              int64
Phone_Service                              int64
Paperless_Billing                          int64
Monthly_Charges                          float64
Total_Charges                            float64
Churn_Value                                int64
Multiple_Lines_No_phone_service             bool
Multiple_Lines_Yes                          bool
Internet_Service_Fiber_optic                bool
Internet_Service_No                         bool
Online_Security_No_internet_service         bool
Online_Se

In [0]:
from pyspark.sql import SparkSession

feature_spark_df = spark.createDataFrame(df)

feature_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("telco_churn_features")

print(f"✓ Feature table written to Delta: telco_churn_features")
print(f"✓ Rows: {df.shape[0]}")
print(f"✓ Columns: {df.shape[1]}")

✓ Feature table written to Delta: telco_churn_features
✓ Rows: 7043
✓ Columns: 31
